# METAVCI_WMH_BLOOD — preprocessing pipeline

Turns raw cohort data into three parquet artefacts used by
`01_analysis.ipynb`:

- **`df.parquet`** — cleaned, labelled dataset (non-imputed)
- **`df_imp.parquet`** — imputed version used for structure learning
- **`bn_vars.parquet`** — `VARIABLE NAME` → `LAYER` metadata

Generic transforms come from the shared `core/` package. The cells
labelled **TODO** are where you decide cohort-specific logic
(which variables, which outcomes, which layers).


## Setup

Make `core/` importable regardless of where this notebook is opened
from (works whether you run from the repo root, the project folder,
or via `jupyter lab`).

In [ ]:
# Cell 1 — put the repo root on sys.path so `import core.*` works.
import sys
from pathlib import Path

_here = Path.cwd().resolve()
_root = _here
while _root != _root.parent and not (_root / "core").is_dir():
    _root = _root.parent
if not (_root / "core").is_dir():
    raise RuntimeError(f"Could not locate `core/` above {_here}")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Repo root: {_root}")


In [ ]:
# Cell 2 — imports.
import logging
import numpy as np
import pandas as pd

from core.config import load_project_config
from core.io import read_sav, apply_value_labels, write_parquet
from core.preprocess import (
    coalesce, to_datetime, impute_dataframe,
    normalise_string_categories, translate_labels, contains_any,
)
# Uncomment if you need SCORE2:
# from core.risk_scores import score2

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)


## Load configuration

In [ ]:
# Cell 3 — read config.yml (path resolution handled by core.config).
config = load_project_config()
print(config)


## Load raw data — **TODO**

Replace the SPSS example below with whatever format your data is in
(CSV, Excel, parquet, ...). If your cohort is split across multiple
files, load and merge them here.


In [ ]:
# Cell 4 — load raw data.
# Example: SPSS baseline + follow-up + merge on patientID.
# df_bl, meta_bl = read_sav(config.raw_dir / "baseline.sav")
# df_fu, meta_fu = read_sav(config.raw_dir / "followup.sav")
# df_bl = apply_value_labels(df_bl, meta_bl)
# df_fu = apply_value_labels(df_fu, meta_fu)
# raw = df_bl.merge(df_fu, on="patientID", how="left")

# Placeholder — replace:
raise NotImplementedError("Load your raw data here and assign it to `raw`.")


## Cohort-specific transforms — **TODO**

Common patterns provided by `core.preprocess`:

- `coalesce(df, cols, target)` — dplyr-style first-non-null across columns
- `to_datetime(df, cols)` — parse date columns
- `contains_any(series, patterns)` — case-insensitive OR-search
- `translate_labels(df, {col: {src: tgt, ...}})` — apply value translations

For anything more elaborate (multi-visit outcome derivation, complex
scoring), write it inline here. Keep this section focused on getting
your columns into the right shape and types.


In [ ]:
# Cell 5 — cohort-specific transforms.
# Example patterns (adapt to your data):

# raw["mean_bp"] = raw[["bp_sys_1", "bp_sys_2"]].mean(axis=1)
# to_datetime(raw, ["visit_date", "outcome_date"])
# coalesce(raw, ["fu1_outcome", "fu2_outcome"], "outcome_any")
# translate_labels(raw, {"sex": {"m": "Male", "f": "Female"}})

df = raw.copy()


## Derive outcomes — **TODO**

Convert follow-up columns into your final outcome variables. Missing
outcomes should be encoded explicitly (e.g. `"Yes" / "No" /
"Unobserved"`) rather than silently imputed to a category — imputing
outcomes changes the estimand.

Example encoding pattern:

```python
mask_unobserved = (df["OUTCOME_X"] == 0) & (df["dropout_reason"] != "no dropout")
df.loc[mask_unobserved, "OUTCOME_X"] = np.nan
df["OUTCOME_X"] = df["OUTCOME_X"].map({1: "Yes", 0: "No"}).fillna("Unobserved").astype("category")
```


In [ ]:
# Cell 6 — derive outcomes.
# TODO: build your outcome columns here.


## Variable selection & layer map — **TODO**

Two options:

1. **From a codebook**: an Excel file with columns
   `VARIABLE NAME` and `LAYER` (matches HBC's convention).
2. **Inline**: build a `bn_vars` DataFrame directly in Python.

`core.bn_utils.build_bn` reads whatever `layer_map` you feed it, so
layer names are up to you. A common structure is
`L0 – Demographics → L2 – Risk factors → ... → L8 – Outcomes → L9 – Dropout`,
but rename freely.

In [ ]:
# Cell 7 — build the bn_vars metadata table.

# Option A: load from codebook Excel
# bn_vars = pd.read_excel(config.codebook_path, sheet_name="Items")
# bn_vars = bn_vars[["LAYER", "VARIABLE NAME"]].dropna(subset=["LAYER"])

# Option B: define inline (small cohorts).
bn_vars = pd.DataFrame([
    # ("L0 – Demographics", "AGE"),
    # ("L0 – Demographics", "SEX"),
    # ("L2 – Risk factors", "SYS_BP"),
    # ("L8 – Outcomes",     "OUTCOME_X"),
    # ("L9 – Dropout",      "DROPOUT REASON"),
], columns=["LAYER", "VARIABLE NAME"])

# Keep only variables actually in df, in the codebook order.
bn_vars = bn_vars[bn_vars["VARIABLE NAME"].isin(df.columns)].reset_index(drop=True)
display(bn_vars)


## Imputation & persistence

`impute_dataframe` uses IterativeImputer for numeric columns and the
modal value for categoricals — matches the HBC pipeline. Outcome
variables should already be encoded categorically (with `"Unobserved"`
where applicable) so they are not silently imputed to a fake value.

In [ ]:
# Cell 8 — impute and write outputs.
analysis_cols = bn_vars["VARIABLE NAME"].tolist()
df_final = df[[c for c in analysis_cols if c in df.columns]].copy()
df_final = normalise_string_categories(df_final)

df_imp = impute_dataframe(df_final, seed=config.seed)

output_dir = config.output_dir
write_parquet(df_final, output_dir / "df.parquet")
write_parquet(df_imp,   output_dir / "df_imp.parquet")
write_parquet(bn_vars,  output_dir / "bn_vars.parquet")

print(f"Wrote {output_dir}/df.parquet         ({len(df_final)} rows, {len(df_final.columns)} cols)")
print(f"Wrote {output_dir}/df_imp.parquet     ({len(df_imp)} rows, {len(df_imp.columns)} cols)")
print(f"Wrote {output_dir}/bn_vars.parquet    ({len(bn_vars)} vars)")
